In [10]:
from transformers import AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model, TaskType

In [11]:
# 1. Initialize a base pre-trained model
model_checkpoint = "distilbert-base-uncased"
base_model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=2)

print("--- Original Model Parameters ---")
print(base_model) 
# Contains standard multi-head attention and linear projection layers

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


--- Original Model Parameters ---
DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
          (ffn): FF

In [12]:
# 2. Define the LoRA Hyperparameter Configuration
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS, # Target task profile (Sequence Classification)
    r=8,                        # Rank: The inner bottleneck size of the adapters
    lora_alpha=16,              # Alpha: A scaling factor that multiplies the learned weights
    lora_dropout=0.1,           # Dropout probability applied specifically to LoRA layers
    target_modules=["q_lin", "v_lin"] # Which internal linear layers to attach sticky notes to
)

In [13]:
# 3. Transform the base model into a parameter-efficient PEFT model
# This freezes the base network and surgically hooks the adapter matrices into place
peft_model = get_peft_model(base_model, lora_config)

print("\n--- LoRA Modified Model Summary ---")
peft_model.print_trainable_parameters()
# Expected Output example: 
# trainable params: 147,456 || all params: 67,100,930 || trainable%: 0.22%


--- LoRA Modified Model Summary ---
trainable params: 739,586 || all params: 67,694,596 || trainable%: 1.0925


In [14]:
# 4. Verify parameter status
# Let's inspect the weights to prove freezing occurred correctly
for name, param in peft_model.named_parameters():
    
    # EXCEPTION: The original reference modules inside the head are kept FROZEN
    if "original_module" in name:
        assert param.requires_grad == False, f"{name} should be frozen!"
        
    # TRAINABLE: Active LoRA adapters AND the active classification head weights
    elif "lora" in name or "modules_to_save" in name or "classifier" in name:
        assert param.requires_grad == True, f"{name} should be trainable!"
        
    # FROZEN: The entire base transformer backbone
    else:
        assert param.requires_grad == False, f"{name} should be frozen!"

print("\nValidation passed: Core backbone and original head references frozen.")
print("Only LoRA adapters and active classification heads accept gradients!")


Validation passed: Core backbone and original head references frozen.
Only LoRA adapters and active classification heads accept gradients!
